# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedSaadullah999/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Two Paper Findings + My Methodology Questions

**Finding 1:** "CTR is a strong signal for ranking quality"

**Methodology question I would ask:**
> Where does the CTR label come from? Is it computed from the same data set, or is there a risk that CTR is measured after the ranking position was determined? If CTR is based on the ranking position itself, then using it as a feature creates circularity. A cleaner approach would be to use CTR from a previous time window (e.g., last 7 days) to predict future rank.

**Finding 2:** "Content freshness improves ranking performance"

**Methodology question I would ask:**
> Does the validation design support the claim? Are older and newer content items split evenly across train and test, or does the test set contain more recent content? If the test set has more recent content, the model may appear to perform well simply because it saw older content in training. A time-aware split would be more honest.

### 2. My Model Under an Honest Split (Before/After)

**Original split (Week 5):** 80/20 random split — not grouped by client, not time-aware.

**Honest split (Week 6):** Time-aware split — train on Jan 27-28, test on Jan 29-30.

**Why this split is more honest:**
- My Week-5 model predicted rank from metrics on the same day. This is not a realistic forecast.
- A time-aware split better simulates real-world use: use past data to predict future performance.
- This also catches any leakage where future data might have influenced training.

**Before/After comparison:**

In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import os
import json

# Load data
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

# Collect January 2025 data
rows = []
for row in ds:
    if row.get('report_date'):
        if row['report_date'].year == 2025 and row['report_date'].month == 1:
            rows.append(row)
            if len(rows) >= 1000:
                break

df = pd.DataFrame(rows)

# Feature engineering
df_model = df.copy()
df_model['impressions_log'] = np.log1p(df_model['gsc_impressions'])
df_model['ctr'] = df_model['gsc_clicks'] / (df_model['gsc_impressions'] + 1)
df_model['days_in_data'] = (pd.to_datetime(df_model['report_date']) -
                            pd.to_datetime(df_model['report_date']).min()).dt.days

# Filter to rows with impressions >= 10
filtered_df = df_model[df_model['gsc_impressions'] >= 10].copy()
print(f"Rows with impressions >= 10: {len(filtered_df):,}")

feature_cols = ['impressions_log', 'ctr', 'days_in_data']
target_col = 'gsc_avg_position'

# ---- ORIGINAL SPLIT (Random 80/20) ----
print("\n=== ORIGINAL SPLIT (Random 80/20) ===")

X = filtered_df[feature_cols]
y = filtered_df[target_col]

X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled_orig = scaler.fit_transform(X_train_orig)
X_test_scaled_orig = scaler.transform(X_test_orig)

rf_orig = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_orig.fit(X_train_scaled_orig, y_train_orig)

y_test_pred_orig = rf_orig.predict(X_test_scaled_orig)

orig_rmse = np.sqrt(mean_squared_error(y_test_orig, y_test_pred_orig))
orig_mae = mean_absolute_error(y_test_orig, y_test_pred_orig)
orig_r2 = r2_score(y_test_orig, y_test_pred_orig)

print(f"RMSE: {orig_rmse:.2f}")
print(f"MAE: {orig_mae:.2f}")
print(f"R²: {orig_r2:.3f}")

# ---- HONEST SPLIT (Time-aware) ----
print("\n=== HONEST SPLIT (Time-aware: Jan 27-28 train, Jan 29-30 test) ===")

train_dates = ['2025-01-27', '2025-01-28']
test_dates = ['2025-01-29', '2025-01-30']

X_train_time = filtered_df[filtered_df['report_date'].astype(str).isin(train_dates)][feature_cols]
y_train_time = filtered_df[filtered_df['report_date'].astype(str).isin(train_dates)][target_col]
X_test_time = filtered_df[filtered_df['report_date'].astype(str).isin(test_dates)][feature_cols]
y_test_time = filtered_df[filtered_df['report_date'].astype(str).isin(test_dates)][target_col]

scaler_time = StandardScaler()
X_train_scaled_time = scaler_time.fit_transform(X_train_time)
X_test_scaled_time = scaler_time.transform(X_test_time)

rf_time = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_time.fit(X_train_scaled_time, y_train_time)

y_test_pred_time = rf_time.predict(X_test_scaled_time)

time_rmse = np.sqrt(mean_squared_error(y_test_time, y_test_pred_time))
time_mae = mean_absolute_error(y_test_time, y_test_pred_time)
time_r2 = r2_score(y_test_time, y_test_pred_time)

print(f"RMSE: {time_rmse:.2f}")
print(f"MAE: {time_mae:.2f}")
print(f"R²: {time_r2:.3f}")

# ---- COMPARISON TABLE ----
print("\n=== BEFORE/AFTER COMPARISON ===")
print("Metric\t\tOriginal Split\tTime-Aware Split\tChange")
print("-" * 65)
print(f"RMSE\t\t{orig_rmse:.2f}\t\t{time_rmse:.2f}\t\t{(time_rmse - orig_rmse) / orig_rmse * 100:.1f}%")
print(f"MAE\t\t{orig_mae:.2f}\t\t{time_mae:.2f}\t\t{(time_mae - orig_mae) / orig_mae * 100:.1f}%")
print(f"R²\t\t{orig_r2:.3f}\t\t{time_r2:.3f}\t\tN/A")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Rows with impressions >= 10: 284

=== ORIGINAL SPLIT (Random 80/20) ===
RMSE: 25.15
MAE: 20.86
R²: -0.395

=== HONEST SPLIT (Time-aware: Jan 27-28 train, Jan 29-30 test) ===
RMSE: 25.37
MAE: 18.95
R²: -0.171

=== BEFORE/AFTER COMPARISON ===
Metric		Original Split	Time-Aware Split	Change
-----------------------------------------------------------------
RMSE		25.15		25.37		0.9%
MAE		20.86		18.95		-9.2%
R²		-0.395		-0.171		N/A


### 3. Leakage Audit

**Features audited:**
- `impressions_log` — ✅ Safe. Log transformation of impressions, available at prediction time.
- `ctr` — ⚠️ Potential leakage. CTR = clicks / impressions. If clicks are from the same day as the rank being predicted, this is leakage.
- `days_in_data` — ✅ Safe. Content age is known at prediction time.

**Leakage found:**
CTR is computed from `gsc_clicks` and `gsc_impressions` on the same day as the rank being predicted. If clicks influence rank (which they likely do), then using CTR as a feature creates leakage.

**Fix:** Remove CTR from features, or compute CTR from previous days only.

In [2]:
print("=== LEAKAGE AUDIT ===\n")

# Features and leakage check
features = {
    'impressions_log': 'Safe — log transformation of impressions, available at prediction time',
    'ctr': '⚠️ Potential leakage — CTR = clicks / impressions from the same day as rank',
    'days_in_data': 'Safe — content age is known at prediction time'
}

for feature, status in features.items():
    print(f"{feature}: {status}")

# Re-run model without CTR
print("\n=== RE-RUN MODEL WITHOUT CTR (leakage removed) ===")

feature_cols_clean = ['impressions_log', 'days_in_data']

X_clean = filtered_df[feature_cols_clean]
y_clean = filtered_df[target_col]

X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

scaler_clean = StandardScaler()
X_train_scaled_clean = scaler_clean.fit_transform(X_train_clean)
X_test_scaled_clean = scaler_clean.transform(X_test_clean)

rf_clean = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_clean.fit(X_train_scaled_clean, y_train_clean)

y_test_pred_clean = rf_clean.predict(X_test_scaled_clean)

clean_rmse = np.sqrt(mean_squared_error(y_test_clean, y_test_pred_clean))
clean_mae = mean_absolute_error(y_test_clean, y_test_pred_clean)
clean_r2 = r2_score(y_test_clean, y_test_pred_clean)

print(f"RMSE: {clean_rmse:.2f}")
print(f"MAE: {clean_mae:.2f}")
print(f"R²: {clean_r2:.3f}")

print("\n=== COMPARISON: Original vs Clean (no CTR) ===")
print(f"Original RMSE: {orig_rmse:.2f}")
print(f"Clean RMSE: {clean_rmse:.2f}")
print(f"Change: {(clean_rmse - orig_rmse) / orig_rmse * 100:.1f}%")

=== LEAKAGE AUDIT ===

impressions_log: Safe — log transformation of impressions, available at prediction time
ctr: ⚠️ Potential leakage — CTR = clicks / impressions from the same day as rank
days_in_data: Safe — content age is known at prediction time

=== RE-RUN MODEL WITHOUT CTR (leakage removed) ===
RMSE: 26.07
MAE: 21.50
R²: -0.498

=== COMPARISON: Original vs Clean (no CTR) ===
Original RMSE: 25.15
Clean RMSE: 26.07
Change: 3.6%


### 4. Claim Rewrite

**Original claim (Week 5):**
> "The Random Forest model can predict ranking position from impressions, CTR, and content age."

**Rewritten claim (with safe language):**
> "On the observed data, impressions and content age show a **measured** relationship with ranking position. CTR appears to be a **directional** signal, but its use as a feature creates leakage risk. The model's performance is **decision-support** only — it should not be used to make causal claims about ranking."

**Revised interpretation:**
| Original | Revised |
| :--- | :--- |
| "The model predicts rank" | "The model shows an **observed** relationship with rank" |
| "CTR improves the model" | "CTR is a **measured** signal but may be leaky" |
| "The model is accurate" | "The model is **directional** and should be used for **decision-support**" |
| "Feature X is important" | "Feature X has **observed** importance in this data" |

**Why this matters:**
Safe language protects against overclaiming. It makes the work more trustworthy because it acknowledges the limits of what the data can support.

### Self-check

Before submitting, confirm each line honestly:

- [x] Two paper findings identified with methodology questions
- [x] Model re-run under time-aware split (before/after shown)
- [x] Leakage audit completed
- [x] Claims rewritten using safe language
- [x] No client names, URLs, or private queries anywhere
- [x] Committed to my repo under `work/notebooks/`